# 3D Gaussian Splatting Evaluation Metrics

3DGS 모델의 품질을 평가하기 위해 세 가지 메트릭을 사용합니다: SSIM, PSNR, LPIPS

## Evaluation 워크플로우

### 명령어

In [ ]:
# 1. Train/test split으로 훈련
python train.py -s <dataset_path> --eval

# 2. 렌더링 생성
python render.py -m <model_path>

# 3. 메트릭 계산
python metrics.py -m <model_path>



### 실행 예시

```
(gaussian_splatting) PS D:\...\gaussian-splatting> python .\metrics.py -m .\output\40e302a5-7\

Scene: .\output\40e302a5-7\
Method: ours_30000
Metric evaluation progress: 100%|██████████████████████| 3/3 [00:02<00:00,  1.30it/s]
  SSIM :    0.9570588
  PSNR :   29.7343693
  LPIPS:    0.1428888
```



**결과 해석**:
- **SSIM 0.957**: 매우 우수한 구조적 유사도 (0.9 이상)
- **PSNR 29.73 dB**: 좋은 품질 (25-30 dB 범위)
- **LPIPS 0.143**: 좋은 지각적 품질 (0.1-0.2 범위)

### 출력 파일

metrics.py 실행 후 다음 파일들이 생성됩니다:

```
output/<model_id>/
├── results.json      # 전체 평균 메트릭
└── per_view.json     # 각 이미지별 메트릭
```



**results.json 예시**:

In [ ]:
{
  "ours_30000": {
    "SSIM": 0.9570588,
    "PSNR": 29.7343693,
    "LPIPS": 0.1428888
  }
}



---

## 1. SSIM (Structural Similarity Index Measure)

### 개념
**구조적 유사도**를 측정하는 지표로, 인간의 시각 시스템이 이미지의 구조적 정보에 민감하다는 원리에 기반합니다.

### 수학적 정의

SSIM은 세 가지 요소의 곱으로 계산됩니다:

$$
\text{SSIM}(x, y) = [l(x,y)]^\alpha \cdot [c(x,y)]^\beta \cdot [s(x,y)]^\gamma
$$

여기서:
- $l(x,y)$: **Luminance comparison** (밝기 비교)
- $c(x,y)$: **Contrast comparison** (대비 비교)
- $s(x,y)$: **Structure comparison** (구조 비교)

일반적으로 $\alpha = \beta = \gamma = 1$을 사용하여 간소화:

$$
\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}
$$

**변수 설명:**
- $\mu_x$, $\mu_y$: 이미지 $x$, $y$의 평균 (mean)
- $\sigma_x^2$, $\sigma_y^2$: 분산 (variance)
- $\sigma_{xy}$: 공분산 (covariance)
- $C_1$, $C_2$: 안정성을 위한 작은 상수 (division by zero 방지)
  - $C_1 = (K_1 L)^2$, $C_2 = (K_2 L)^2$
  - $L$: 픽셀 값의 dynamic range (보통 255 for 8-bit images)
  - $K_1 = 0.01$, $K_2 = 0.03$ (일반적인 값)

### 특징
- **범위**: 0 (완전히 다름) ~ 1 (동일함)
- **대칭성**: $\text{SSIM}(x, y) = \text{SSIM}(y, x)$
- **국소적 계산**: 보통 11×11 Gaussian window를 사용하여 sliding window 방식으로 계산
- **인간 지각과의 상관성**: 단순 픽셀 차이보다 인간의 시각적 품질 평가와 더 잘 일치

### Python 구현

3DGS에서 실제 사용하는 SSIM 구현 (`utils/loss_utils.py`):

In [ ]:
# utils/loss_utils.py
import torch
import torch.nn.functional as F
from torch.autograd import Variable
from math import exp

def gaussian(window_size, sigma):
    gauss = torch.Tensor([
        exp(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2)) 
        for x in range(window_size)
    ])
    return gauss / gauss.sum()

def create_window(window_size, channel):
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = Variable(_2D_window.expand(channel, 1, window_size, window_size).contiguous())
    return window

def ssim(img1, img2, window_size=11, size_average=True):
    """
    SSIM 계산
    
    Args:
        img1, img2: [B, C, H, W] 형태의 텐서 (0-1 범위)
        window_size: Gaussian window 크기 (default: 11)
        size_average: True면 평균, False면 합
    
    Returns:
        SSIM 값 (scalar)
    """
    channel = img1.size(-3)
    window = create_window(window_size, channel)

    if img1.is_cuda:
        window = window.cuda(img1.get_device())
    window = window.type_as(img1)

    return _ssim(img1, img2, window, window_size, channel, size_average)

def _ssim(img1, img2, window, window_size, channel, size_average=True):
    # 평균 계산 (μx, μy)
    mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channel)
    mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channel)

    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2

    # 분산과 공분산 계산 (σx², σy², σxy)
    sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channel) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channel) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channel) - mu1_mu2

    # 상수 정의 (0-1 범위 이미지 기준)
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    # SSIM 계산
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

    if size_average:
        return ssim_map.mean()
    else:
        return ssim_map.mean(1).mean(1).mean(1)



> **참고**: 3DGS는 CUDA 가속화된 `fusedssim`도 제공합니다 (`diff_gaussian_rasterization`에 포함).

### 해석
- **0.9 이상**: 매우 우수한 품질
- **0.8 ~ 0.9**: 좋은 품질
- **0.7 ~ 0.8**: 보통 품질
- **0.7 미만**: 낮은 품질

---

## 2. PSNR (Peak Signal-to-Noise Ratio)

### 개념
신호(원본 이미지)의 최대 가능한 power와 noise(오차)의 power 비율을 데시벨(dB) 단위로 표현한 지표입니다.

### 수학적 정의

먼저 **MSE (Mean Squared Error)**를 계산:

$$
\text{MSE} = \frac{1}{HW} \sum_{i=1}^{H} \sum_{j=1}^{W} [I(i,j) - K(i,j)]^2
$$

그 다음 PSNR을 계산:

$$
\text{PSNR} = 10 \cdot \log_{10}\left(\frac{\text{MAX}_I^2}{\text{MSE}}\right) = 20 \cdot \log_{10}\left(\frac{\text{MAX}_I}{\sqrt{\text{MSE}}}\right)
$$

**변수 설명:**
- $I$: 원본 이미지 (ground truth)
- $K$: 재구성된 이미지 (rendered)
- $H, W$: 이미지 높이와 너비
- $\text{MAX}_I$: 픽셀의 최대 가능 값
  - 8-bit 이미지: $\text{MAX}_I = 255$
  - Float [0, 1] 범위: $\text{MAX}_I = 1.0$

### 특징
- **범위**: 보통 20 ~ 40 dB (높을수록 좋음)
- **무한대**: MSE = 0일 때 (완벽한 재구성)
- **픽셀 레벨**: 픽셀 단위의 절대 오차를 측정
- **대역폭 효율**: 원래 이미지 압축 품질 평가에 사용되었으나, 3D reconstruction에도 널리 사용

### Python 구현

3DGS에서 실제 사용하는 PSNR 구현 (`utils/image_utils.py`):

In [ ]:
# utils/image_utils.py
import torch

def mse(img1, img2):
    """
    Mean Squared Error 계산
    
    Args:
        img1, img2: [B, C, H, W] 형태의 이미지
    
    Returns:
        MSE 값 [B, 1]
    """
    return (((img1 - img2)) ** 2).view(img1.shape[0], -1).mean(1, keepdim=True)

def psnr(img1, img2):
    """
    PSNR 계산
    
    Args:
        img1, img2: [B, C, H, W] 이미지 텐서 (0-1 범위)
    
    Returns:
        PSNR 값 (dB) [B, 1]
    """
    mse_val = (((img1 - img2)) ** 2).view(img1.shape[0], -1).mean(1, keepdim=True)
    return 20 * torch.log10(1.0 / torch.sqrt(mse_val))



> **참고**: 3DGS는 `max_val=1.0` (0-1 범위 이미지)을 사용합니다.

### 해석
- **> 35 dB**: 매우 우수한 품질 (거의 손실 없음)
- **30 ~ 35 dB**: 우수한 품질
- **25 ~ 30 dB**: 좋은 품질 (일반적인 3DGS 결과)
- **20 ~ 25 dB**: 보통 품질
- **< 20 dB**: 낮은 품질 (시각적으로 명확한 차이)

### PSNR의 한계
PSNR은 픽셀 단위 차이만 측정하므로:
- 작은 공간적 shift에도 큰 패널티
- 구조적 유사성을 고려하지 않음
- 인간의 시각적 지각과 항상 일치하지는 않음

→ 이를 보완하기 위해 SSIM과 함께 사용

---

## 3. LPIPS (Learned Perceptual Image Patch Similarity)

### 개념
**딥러닝 네트워크의 feature space**에서 이미지 유사도를 측정하는 지표입니다. VGG, AlexNet 등의 pre-trained network가 학습한 feature가 인간의 지각적 유사도를 더 잘 반영한다는 아이디어에 기반합니다.

### 동작 원리

1. **Pre-trained CNN 사용**: VGG16, AlexNet 등
2. **Multi-scale feature extraction**: 여러 layer의 activation 추출
3. **Feature distance 계산**: L2 distance in feature space
4. **Weighted sum**: 각 layer의 중요도로 가중 평균

### 수학적 정의

두 이미지 $x$, $x_0$에 대해:

$$
d(x, x_0) = \sum_l \frac{1}{H_l W_l} \sum_{h,w} ||w_l \odot (\hat{y}_l^{h,w} - \hat{y}_{0,l}^{h,w})||_2^2
$$

**변수 설명:**
- $\hat{y}_l$, $\hat{y}_{0,l}$: Layer $l$에서 추출된 normalized feature maps
- $H_l, W_l$: Layer $l$의 feature map 크기
- $w_l$: Layer $l$의 학습된 가중치
- $\odot$: Element-wise multiplication

### Feature Normalization

각 layer의 feature는 channel-wise로 정규화됩니다:

$$
\hat{y}_{l}^{h,w} = \frac{y_{l}^{h,w}}{||\sum_{h,w} y_{l}^{h,w}||_2}
$$

### 특징
- **범위**: 0 (동일) ~ 1 (완전히 다름) (이론적으로 unbounded이지만 실제로는 0-1)
- **Perceptual**: 인간의 지각적 유사도와 높은 상관관계
- **Learned**: 대규모 데이터셋에서 학습된 feature 사용
- **Multi-scale**: 여러 layer의 정보를 종합

### Python 구현 (간소화)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class LPIPS(nn.Module):
    def __init__(self, net='vgg'):
        super(LPIPS, self).__init__()
        
        if net == 'vgg':
            # VGG16 pre-trained model 로드
            self.net = models.vgg16(pretrained=True).features
            self.layers = [3, 8, 15, 22, 29]  # relu1_2, relu2_2, relu3_3, relu4_3, relu5_3
        
        # Feature extraction만 사용 (gradient 불필요)
        for param in self.net.parameters():
            param.requires_grad = False
        
        # Linear layer weights (학습됨)
        self.lins = nn.ModuleList([
            nn.Conv2d(64, 1, 1, bias=False),
            nn.Conv2d(128, 1, 1, bias=False),
            nn.Conv2d(256, 1, 1, bias=False),
            nn.Conv2d(512, 1, 1, bias=False),
            nn.Conv2d(512, 1, 1, bias=False),
        ])
        
    def forward(self, img1, img2):
        """
        LPIPS distance 계산
        
        Args:
            img1, img2: [B, 3, H, W] 텐서 (0-1 범위)
        
        Returns:
            LPIPS distance (낮을수록 유사)
        """
        # Normalize to ImageNet statistics
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(img1.device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(img1.device)
        
        img1 = (img1 - mean) / std
        img2 = (img2 - mean) / std
        
        # Extract features
        feats1 = self.extract_features(img1)
        feats2 = self.extract_features(img2)
        
        # Compute distances
        dists = []
        for i, (f1, f2) in enumerate(zip(feats1, feats2)):
            # Normalize features
            f1_norm = f1 / (torch.sqrt(torch.sum(f1**2, dim=1, keepdim=True)) + 1e-10)
            f2_norm = f2 / (torch.sqrt(torch.sum(f2**2, dim=1, keepdim=True)) + 1e-10)
            
            # Compute distance
            diff = (f1_norm - f2_norm) ** 2
            
            # Apply learned weights
            dist = self.lins[i](diff)
            
            # Spatial average
            dist = dist.mean(dim=[2, 3], keepdim=True)
            dists.append(dist)
        
        # Sum across layers
        return torch.sum(torch.cat(dists, dim=0))
    
    def extract_features(self, x):
        """Extract multi-scale features from VGG"""
        feats = []
        for i, layer in enumerate(self.net):
            x = layer(x)
            if i in self.layers:
                feats.append(x)
        return feats

# 사용 예시
lpips_metric = LPIPS(net='vgg').eval()

img1 = torch.rand(1, 3, 256, 256)
img2 = torch.rand(1, 3, 256, 256)

with torch.no_grad():
    distance = lpips_metric(img1, img2)
    print(f"LPIPS distance: {distance.item():.4f}")



### 3DGS에서의 실제 사용

3DGS의 `metrics.py`에서는 `lpipsPyTorch` 래퍼를 사용합니다:

In [ ]:
# metrics.py에서의 LPIPS 사용
from lpipsPyTorch import lpips

# 각 이미지쌍에 대해 LPIPS 계산
lpips_value = lpips(render_tensor, gt_tensor, net_type='vgg')



**lpipsPyTorch 래퍼** (`lpipsPyTorch/__init__.py`):

In [ ]:
import lpips

# 전역 모델 캐싱
loss_fn_alex = None
loss_fn_vgg = None

def lpips(img1, img2, net_type='alex', version='0.1'):
    global loss_fn_alex, loss_fn_vgg
    
    if net_type == 'alex':
        if loss_fn_alex is None:
            loss_fn_alex = lpips.LPIPS(net=net_type, version=version)
            if img1.is_cuda:
                loss_fn_alex.cuda()
        return loss_fn_alex(img1, img2)
    elif net_type == 'vgg':
        if loss_fn_vgg is None:
            loss_fn_vgg = lpips.LPIPS(net=net_type, version=version)
            if img1.is_cuda:
                loss_fn_vgg.cuda()
        return loss_fn_vgg(img1, img2)



> **참고**: LPIPS 라이브러리는 내부적으로 이미지를 [-1, 1] 범위로 정규화합니다.

### VGG16 Architecture (LPIPS에 사용되는 layers)

```
Layer 3  (relu1_2): 64 channels   - Low-level features (edges, colors)
Layer 8  (relu2_2): 128 channels  - Textures, simple patterns
Layer 15 (relu3_3): 256 channels  - Object parts
Layer 22 (relu4_3): 512 channels  - Object-level features
Layer 29 (relu5_3): 512 channels  - High-level semantic features
```



### 해석
- **< 0.1**: 매우 우수한 지각적 품질
- **0.1 ~ 0.2**: 좋은 지각적 품질
- **0.2 ~ 0.3**: 보통 품질
- **> 0.3**: 낮은 품질 (사람 눈에 명확한 차이)

### LPIPS의 장점
1. **인간 지각과 높은 상관성**: MSE, PSNR보다 인간의 품질 평가와 더 잘 일치
2. **Shift invariance**: 작은 공간적 이동에 덜 민감
3. **구조적 유사성**: 픽셀 단위가 아닌 feature 단위로 비교
4. **학습 기반**: 대규모 데이터로 학습되어 일반화 능력 우수

---

## 종합 비교

| Metric | 범위 | 방향 | 측정 대상 | 장점 | 단점 |
|--------|------|------|-----------|------|------|
| **SSIM** | 0-1 | ↑ | 구조적 유사도 | 인간 지각 고려, 빠름 | 국소적 윈도우 기반 |
| **PSNR** | dB | ↑ | 픽셀 오차 | 간단, 해석 용이 | 구조 무시, 지각 불일치 |
| **LPIPS** | 0-1 | ↓ | 지각적 유사도 | 인간 지각 최적, 강건 | 느림, 네트워크 의존 |

### 사용 권장사항

1. **SSIM**: 구조적 품질 평가, 빠른 피드백
2. **PSNR**: 픽셀 레벨 정확도, 정량적 비교
3. **LPIPS**: 최종 품질 평가, 인간 지각 중요시

**Best Practice**: 세 가지 메트릭을 모두 사용하여 종합적으로 평가

---

## 실전 예제: 3DGS metrics.py

### metrics.py 핵심 구조

실제 3DGS의 `metrics.py`는 다음과 같이 동작합니다:

In [ ]:
# metrics.py (simplified)
from pathlib import Path
import os
import torch
import torchvision.transforms.functional as tf
from PIL import Image
from tqdm import tqdm
import json

from utils.loss_utils import ssim
from utils.image_utils import psnr
from lpipsPyTorch import lpips

def readImages(renders_dir, gt_dir):
    """렌더링 이미지와 GT 이미지 로드"""
    renders = []
    gts = []
    image_names = []
    
    for fname in os.listdir(renders_dir):
        render = Image.open(renders_dir / fname)
        gt = Image.open(gt_dir / fname)
        
        # [B, C, H, W] 텐서로 변환, :3으로 RGB만 사용 (alpha 제외)
        renders.append(tf.to_tensor(render).unsqueeze(0)[:, :3, :, :].cuda())
        gts.append(tf.to_tensor(gt).unsqueeze(0)[:, :3, :, :].cuda())
        image_names.append(fname)
    
    return renders, gts, image_names

def evaluate(model_paths):
    """모델 평가 실행"""
    for scene_dir in model_paths:
        print("Scene:", scene_dir)
        
        test_dir = Path(scene_dir) / "test"
        
        for method in os.listdir(test_dir):
            print("Method:", method)
            
            method_dir = test_dir / method
            gt_dir = method_dir / "gt"
            renders_dir = method_dir / "renders"
            
            renders, gts, image_names = readImages(renders_dir, gt_dir)
            
            ssims = []
            psnrs = []
            lpipss = []
            
            for idx in tqdm(range(len(renders)), desc="Metric evaluation progress"):
                ssims.append(ssim(renders[idx], gts[idx]))
                psnrs.append(psnr(renders[idx], gts[idx]))
                lpipss.append(lpips(renders[idx], gts[idx], net_type='vgg'))
            
            # 결과 출력
            print("  SSIM : {:>12.7f}".format(torch.tensor(ssims).mean()))
            print("  PSNR : {:>12.7f}".format(torch.tensor(psnrs).mean()))
            print("  LPIPS: {:>12.7f}".format(torch.tensor(lpipss).mean()))
            
            # JSON으로 저장
            results = {
                method: {
                    "SSIM": torch.tensor(ssims).mean().item(),
                    "PSNR": torch.tensor(psnrs).mean().item(),
                    "LPIPS": torch.tensor(lpipss).mean().item()
                }
            }
            
            with open(scene_dir + "/results.json", 'w') as fp:
                json.dump(results, fp, indent=True)



### 폴더 구조

`render.py` 실행 후 생성되는 구조:

```
output/<model_id>/
├── test/
│   └── ours_30000/          # iteration 30000의 결과
│       ├── gt/              # Ground truth 이미지
│       │   ├── 00000.png
│       │   ├── 00001.png
│       │   └── ...
│       └── renders/         # 렌더링된 이미지
│           ├── 00000.png
│           ├── 00001.png
│           └── ...
├── results.json             # metrics.py 실행 후 생성
└── per_view.json            # 각 이미지별 메트릭
```



---

## 참고 자료

### Papers
1. **SSIM**: Wang et al. "Image Quality Assessment: From Error Visibility to Structural Similarity" (2004)
2. **LPIPS**: Zhang et al. "The Unreasonable Effectiveness of Deep Features as a Perceptual Metric" (2018)
3. **3DGS**: Kerbl et al. "3D Gaussian Splatting for Real-Time Radiance Field Rendering" (2023)

### Libraries
- **SSIM**: `scikit-image`, `pytorch-msssim`
- **LPIPS**: `lpips` (official implementation)
- **Metrics**: Standard in PyTorch, NumPy

### 추가 메트릭
실제 연구에서는 추가 메트릭도 사용:
- **MS-SSIM**: Multi-scale SSIM (여러 해상도에서 SSIM 계산)
- **FID**: Fréchet Inception Distance (생성 모델 평가)
- **KID**: Kernel Inception Distance
- **Depth Metrics**: Depth accuracy, completeness (depth supervision 사용 시)

---

## 결론

3DGS 모델 평가에는 세 가지 메트릭이 상호 보완적으로 사용됩니다:

1. **PSNR**: 기본적인 픽셀 정확도
2. **SSIM**: 구조적 품질
3. **LPIPS**: 지각적 품질 (인간 평가와 가장 유사)

좋은 3DGS 모델은 세 가지 메트릭 모두에서 우수한 성능을 보여야 하며, 특히 **LPIPS**가 실제 시각적 품질과 가장 잘 상관됩니다.